### Translate R code from Muff et al. (2019) SSF otter habitat selection analysis into Python

In [1]:
#### code is run on GPU CPOM server for speed
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.patches as mpatches
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from matplotlib import gridspec
import os
import warnings
warnings.filterwarnings("ignore")

WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


In [2]:
# ### model function

def run_SSF(dat):
    # create unique animal ID
    dat["ANIMAL_ID"] = dat["ID"].astype("category").cat.codes + 1

    # create stratum ID mapping
    unique_strata = dat["ID"].unique()
    strata_map = dict(zip(unique_strata, range(1, len(unique_strata) + 1)))
    dat["str_ID"] = dat["ID"].map(strata_map)

    # sort by stratum ID
    dat = dat.sort_values(by="str_ID")

    # standardize continuous variables
    scaler = StandardScaler()
    dat["d_to_land"] = scaler.fit_transform(dat[["d_to_land"]])
    dat["step_length"] = scaler.fit_transform(dat[["step_length"]])

    ### random effects model
    coords = {
        "observations": dat.index,
        "strata": dat["str_ID"].unique(),
        "animals": dat["ANIMAL_ID"].unique(),
    }

    with pm.Model(coords=coords) as model:
        # load IDs
        str_id = pm.Data("str_ID", dat["str_ID"].values)
        animal_id = pm.Data("animal_ID", dat["ANIMAL_ID"].values)

        # load covariates
        SIC = pm.Data("sic", dat["sic"].values)
        SIT = pm.Data("sit", dat["sit"].values) 
        SR = pm.Data("sr", dat["sr"].values)
        SD = pm.Data("sd", dat["sd"].values)
        distance = pm.Data("d_to_land", dat["d_to_land"].values)
        steps = pm.Data("step_length", dat["step_length"].values)

        # fixed Effects (no intercept)
        beta_SIC = pm.Normal("beta_SIC", mu=0, sigma=10)
        beta_SIT = pm.Normal("beta_SIT", mu=0, sigma=10)
        beta_SD = pm.Normal("beta_SD", mu=0, sigma=10)
        beta_SR = pm.Normal("beta_SR", mu=0, sigma=10)
        beta_distance = pm.Normal("beta_distance", mu=0, sigma=10)
        beta_steps = pm.Normal("beta_steps", mu=0, sigma=10)

        # random intercept per stratum (mimicking clogit)
        sigma_stratum = pm.Exponential("sigma_stratum", 1.0)
        stratum_re = pm.Normal("stratum_re", mu=0, sigma=sigma_stratum, dims="strata")

        # random slopes per animal for distance to land, SIC, chosen variable
        sigma_distance = pm.Exponential("sigma_distance", 1.0)
        sigma_sic = pm.Exponential("sigma_sic", 1.0)
        sigma_sit = pm.Exponential("sigma_sit", 1.0)
        sigma_sd = pm.Exponential("sigma_sd", 1.0)
        sigma_sr = pm.Exponential("sigma_sr", 1.0)

        re_distance = pm.Normal("re_distance", mu=0, sigma=sigma_distance, dims="animals")
        re_sic = pm.Normal("re_sic", mu=0, sigma=sigma_sic, dims="animals")
        re_sit = pm.Normal("re_sit", mu=0, sigma=sigma_sit, dims="animals")
        re_sd = pm.Normal("re_sd", mu=0, sigma=sigma_sd, dims="animals")
        re_sr = pm.Normal("re_sr", mu=0, sigma=sigma_sr, dims="animals")

        # linear predictor
        eta = (
            beta_SIC * SIC +
            beta_SIT * SIT +
            beta_SR * SR +
            beta_SD * SD +
            beta_distance * distance +
            beta_steps * steps +
            stratum_re[str_id - 1] +  # Adjust for 0-indexing
            re_distance[animal_id - 1] * distance +
            re_sic[animal_id - 1] * SIC +
            re_sit[animal_id - 1] * SIT +
            re_sr[animal_id - 1] * SR +
            re_sd[animal_id - 1] * SD
        )
        # poisson likelihood
        mu = pm.math.exp(eta)
        Loc_obs = pm.Poisson("Loc_obs", mu=mu, observed=dat["case"].values, dims="observations")

        # sample
        trace = pm.sample(1000, tune=1000, target_accept=0.99, return_inferencedata=True)
  
        return trace  

In [3]:
### run SSF 
datapath_out = '/home/cjn/pb_new/SSF/1.1km/results/'
months = [4]
month_names = ['Apr']

datapath = f'/home/cjn/pb_new/SSF/1.1km/random_steps/'
files = glob.glob(f'{datapath}*.csv')

for month, month_name in zip(months,month_names):
    filepath_out = f'{datapath_out}/{month}.csv'
        
    if os.path.exists(filepath_out) == False:
        dfs = []
        for file in files:
            all_data = pd.read_csv(file, index_col=None, header=0).dropna()
            all_data['month'] = [int(date[5:7]) for date in all_data['date']]
            month_data = all_data[all_data['month']==month]

            dfs.append(month_data)

        dat = pd.concat(dfs, axis=0, ignore_index=True)
        try:
            trace = run_SSF(dat)
            d = az.summary(trace, var_names=["beta_SIC", "beta_SIT", "beta_SD", "beta_SR",  "beta_distance", "beta_steps"])
            d.to_csv(filepath_out)
        except:
            print(month,'fail')